In [1]:
import math


# ------------------------------------------------
# 1. Example Dataset
# ------------------------------------------------

# Each example contains attributes and target value
examples = [
    {"color": "red",   "shape": "round",   "size": "small", "target": True},
    {"color": "red",   "shape": "square",  "size": "small", "target": True},
    {"color": "blue",  "shape": "round",   "size": "small", "target": False},
    {"color": "blue",  "shape": "square",  "size": "large", "target": False},
    {"color": "green", "shape": "round",   "size": "large", "target": False},
    {"color": "red",   "shape": "round",   "size": "large", "target": True},
    {"color": "red",   "shape": "triangle","size": "large", "target": True},
    {"color": "blue",  "shape": "triangle","size": "small", "target": False},
]

In [2]:
Pos = [
    example for example in examples
    if example["target"] == True
]

Neg = [
    example for example in examples
    if example["target"] == False
]


# ------------------------------------------------
# 3. Candidate Literals
# ------------------------------------------------

def generate_candidate_literals(examples):
    """
    Generate possible literals from the attributes
    in the dataset.
    """

    literals = []

    attributes = ["color", "shape", "size"]

    for attribute in attributes:

        values = set(example[attribute] for example in examples)

        for value in values:

            literals.append((attribute, value))

    return literals


In [3]:
def satisfies(example, literal):

    attribute, value = literal

    return example[attribute] == value

In [4]:
def satisfies_rule(example, rule):

    for literal in rule:

        if not satisfies(example, literal):
            return False

    return True

In [5]:
def covered_examples(examples, rule):

    return [
        example for example in examples
        if satisfies_rule(example, rule)
    ]
    

In [6]:
def foil_gain(literal, rule, Pos, Neg):

    """
    Calculate FOIL Gain for a candidate literal.

    Gain(L) =
        t * [log2(p1/(p1+n1)) - log2(p0/(p0+n0))]

    where:
        p0 = positive examples before adding literal
        n0 = negative examples before adding literal
        p1 = positive examples after adding literal
        n1 = negative examples after adding literal
        t  = positive examples covered after adding literal
    """

    # Examples covered by current rule
    old_pos = covered_examples(Pos, rule)
    old_neg = covered_examples(Neg, rule)

    p0 = len(old_pos)
    n0 = len(old_neg)

    # If there are no positive examples,
    # gain is zero
    if p0 == 0:
        return 0

    new_rule = rule + [literal]

    new_pos = covered_examples(Pos, new_rule)
    new_neg = covered_examples(Neg, new_rule)

    p1 = len(new_pos)
    n1 = len(new_neg)

    # If candidate covers no positive examples
    if p1 == 0:
        return 0

    # Avoid division by zero
    if p0 + n0 == 0:
        return 0

    if p1 + n1 == 0:
        return 0

    # FOIL probabilities
    old_probability = p0 / (p0 + n0)
    new_probability = p1 / (p1 + n1)

    # If probability does not improve
    if old_probability == 0 or new_probability == 0:
        return 0

    # FOIL Gain
    t = p1

    gain = t * (
        math.log2(new_probability)
        - math.log2(old_probability)
    )

    return gain

In [8]:
def learn_new_rule(Pos, Neg, all_examples):

    """
    Corresponds to:

        NewRule <- rule predicting Target-predicate
                   with no preconditions

        while NewRuleNeg:
            Add a new literal
    """

    # Start with an empty rule
    new_rule = []

    # Negative examples currently covered
    new_rule_neg = covered_examples(Neg, new_rule)

    used_literals = set()

    print("\nLearning a new rule...")

    while len(new_rule_neg) > 0:

        # Generate candidate literals
        candidate_literals = generate_candidate_literals(all_examples)

        # Remove already-used literals
        candidate_literals = [
            literal
            for literal in candidate_literals
            if literal not in used_literals
        ]

        # If no literals remain
        if not candidate_literals:
            break

        # Calculate FOIL gain for every candidate
        gains = {}

        for literal in candidate_literals:

            gain = foil_gain(
                literal,
                new_rule,
                Pos,
                Neg
            )

            gains[literal] = gain

        # Select literal with maximum gain
        best_literal = max(
            gains,
            key=gains.get
        )

        best_gain = gains[best_literal]

        print(
            f"Candidate: {best_literal} "
            f"-> Gain = {best_gain:.4f}"
        )

        # If no useful literal is found
        if best_gain <= 0:
            break

        # Add best literal to rule
        new_rule.append(best_literal)

        used_literals.add(best_literal)
        
        new_rule_neg = covered_examples(
            Neg,
            new_rule
        )

        print(
            f"Added literal: {best_literal}"
        )

        print(
            f"Remaining negative examples: "
            f"{len(new_rule_neg)}"
        )

    return new_rule

In [9]:
def FOIL(examples):

    """
    Main FOIL algorithm.

    Corresponds to:

        Pos <- positive examples
        Neg <- negative examples
        Learned_rules <- {}

        while Pos:

            NewRule <- Learn a New Rule

            Learned_rules <- Learned_rules + NewRule

            Pos <- Pos - examples covered by NewRule
    """

    # Positive examples
    Pos = [
        example for example in examples
        if example["target"] == True
    ]

    # Negative examples
    Neg = [
        example for example in examples
        if example["target"] == False
    ]

    learned_rules = []

    print("===================================")
    print("           FOIL ALGORITHM")
    print("===================================")

    print(f"\nPositive examples: {len(Pos)}")
    print(f"Negative examples: {len(Neg)}")

    # Continue until all positive examples
    # are covered
    while len(Pos) > 0:

        print("\n-----------------------------------")
        print("Remaining Positive Examples:",
              len(Pos))
        print("-----------------------------------")

        # Learn a new rule
        new_rule = learn_new_rule(
            Pos,
            Neg,
            examples
        )

        # If no rule can be learned
        if len(new_rule) == 0:
            print("\nNo more useful rules can be found.")
            break

        # Store learned rule
        learned_rules.append(new_rule)

        print("\nNew Rule Learned:")
        print(format_rule(new_rule))

        # Find positive examples covered
        covered_pos = covered_examples(
            Pos,
            new_rule
        )

        print(
            f"Positive examples covered: "
            f"{len(covered_pos)}"
        )

        # Remove covered positive examples
        Pos = [
            example
            for example in Pos
            if example not in covered_pos
        ]

    return learned_rules

In [10]:
def format_rule(rule):

    if len(rule) == 0:
        return "Target(X)"

    conditions = []

    for attribute, value in rule:

        conditions.append(
            f"{attribute}(X) = {value}"
        )

    return "Target(X) :- " + " AND ".join(conditions)

In [11]:
learned_rules = FOIL(examples)

           FOIL ALGORITHM

Positive examples: 4
Negative examples: 4

-----------------------------------
Remaining Positive Examples: 4
-----------------------------------

Learning a new rule...
Candidate: ('color', 'red') -> Gain = 4.0000
Added literal: ('color', 'red')
Remaining negative examples: 0

New Rule Learned:
Target(X) :- color(X) = red
Positive examples covered: 4


In [12]:
print("\n\n===================================")
print("       FINAL LEARNED RULES")
print("===================================")

for i, rule in enumerate(learned_rules, start=1):

    print(
        f"Rule {i}: "
        f"{format_rule(rule)}"
    )



       FINAL LEARNED RULES
Rule 1: Target(X) :- color(X) = red


In [13]:

def predict(example, learned_rules):

    for rule in learned_rules:

        if satisfies_rule(example, rule):
            return True

    return False


print("\n===================================")
print("           PREDICTIONS")
print("===================================")

for example in examples:

    prediction = predict(
        example,
        learned_rules
    )

    print(
        f"Color={example['color']:<6} "
        f"Shape={example['shape']:<9} "
        f"Size={example['size']:<6} "
        f"Actual={example['target']} "
        f"Predicted={prediction}"
    )


           PREDICTIONS
Color=red    Shape=round     Size=small  Actual=True Predicted=True
Color=red    Shape=square    Size=small  Actual=True Predicted=True
Color=blue   Shape=round     Size=small  Actual=False Predicted=False
Color=blue   Shape=square    Size=large  Actual=False Predicted=False
Color=green  Shape=round     Size=large  Actual=False Predicted=False
Color=red    Shape=round     Size=large  Actual=True Predicted=True
Color=red    Shape=triangle  Size=large  Actual=True Predicted=True
Color=blue   Shape=triangle  Size=small  Actual=False Predicted=False


In [14]:
# New unseen example

new_example = {
    "color": "red",
    "shape": "round",
    "size": "small"
}

prediction = predict(
    new_example,
    learned_rules
)

print("New Example:")
print(new_example)

print("\nFOIL Prediction:", prediction)

New Example:
{'color': 'red', 'shape': 'round', 'size': 'small'}

FOIL Prediction: True
